In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import seaborn as sns

In [ ]:
# checking shape of individual files
folder = r"Historic Demand 2010 - 2026"  # use r"" to avoid slash issues

# fetch it only for demand files from 2010-2024
for year in range(2010, 2025):
    df = pd.read_csv(f"{folder}/demanddata_{year}.csv")
    print(year, df.shape)

In [ ]:
# checking missing values in individual demand datasets from 2010-2024
for year in range(2010, 2025):
    missingval_by_col = df.isna().sum()
    missingval_by_col = missingval_by_col[missingval_by_col > 0].sort_values(ascending=False)
    print(f"\n{year}  shape={df.shape}")
    print(missingval_by_col if not missingval_by_col.empty else "No missing values")

In [ ]:
for year in range(2010, 2025):
    df = pd.read_csv(f"{folder}/demanddata_{year}.csv", nrows=5)  # just read a few rows
    print(year, df.columns.tolist())

### CONCATENATION

In [ ]:
dfs = []
for year in range(2010, 2025):
    dfs.append(pd.read_csv(f"{folder}/demanddata_{year}.csv"))

df_all = pd.concat(dfs, ignore_index=True)

print("Final shape:", df_all.shape)
print(df_all.head())

In [ ]:
df_all.head(5)

In [ ]:
# check data types
df_all.dtypes

### DATA QUALITY

In [ ]:
# check missing values
df_all.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
df_new = df_all.drop(columns=["EMBEDDED_WIND_CAPACITY", "EMBEDDED_SOLAR_CAPACITY",
                              "IFA_FLOW", "IFA2_FLOW", "BRITNED_FLOW",
                              "MOYLE_FLOW", "EAST_WEST_FLOW", "NEMO_FLOW","NSL_FLOW","ELECLINK_FLOW","VIKING_FLOW","GREENLINK_FLOW","SCOTTISH_TRANSFER"])

In [ ]:
df_new.head(10)

In [ ]:
df_new.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
# Count number of settlement periods (rows) for each date

periods_per_date = df_new.groupby("SETTLEMENT_DATE")["SETTLEMENT_PERIOD"].count()

print("First 10 dates and their period counts:")
print(periods_per_date.head(10))

not_48 = periods_per_date[periods_per_date != 48]

print("\nNumber of dates where period count is NOT 48:", not_48.shape[0])

print("\nFirst 30 dates where period count is NOT 48:")
print(not_48.sort_index().head(30))

In [ ]:
# ensure datetime datatype of SETTLEMENT_DATE
df_new["SETTLEMENT_DATE"] = pd.to_datetime(df_new["SETTLEMENT_DATE"], dayfirst=True, errors="coerce")

# group by year without adding a YEAR column
year_summary = (
    df_new.groupby(df_new["SETTLEMENT_DATE"].dt.year)["SETTLEMENT_DATE"]
          .agg(min_date="min", max_date="max", unique_dates="nunique")
)

print("Date coverage per year:")
print(year_summary)

In [ ]:
# 1) for each day, check if it has 48 settlement periods
periods_per_day = df_new.groupby("SETTLEMENT_DATE")["SETTLEMENT_PERIOD"].count()

print("\nSummary of period counts across all days (e.g., 48, 46, 50):")
print(periods_per_day.value_counts().sort_index())

not_48 = periods_per_day[periods_per_day != 48].sort_index()
print("\nNumber of dates that are NOT 48 periods:", not_48.shape[0])

# 2)  breakdown of 'not 48' dates by year
not_48_by_year = not_48.groupby(not_48.index.year).size()
print("\nHow many 'not 48' days in each year:")
print(not_48_by_year)

In [ ]:
# Check duplicate rows based on (SETTLEMENT_DATE, SETTLEMENT_PERIOD)

dups_count = df_new.duplicated(["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]).sum()
print("Number of duplicate (date, period) rows:", dups_count)

# Show some duplicate examples (if any)
if dups_count > 0:
    dup_rows = df_new[df_new.duplicated(["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"], keep=False)] \
        .sort_values(["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"])
    
    print("\nFirst 20 duplicate rows:")
    print(dup_rows.head(20))

In [ ]:
# Sanity check: Demand should not be negative 
# ND: National Demand
# TSD: Transmission System Demand
print("ND < 0 rows:", (df_new["ND"] < 0).sum())
print("TSD < 0 rows:", (df_new["TSD"] < 0).sum())

print("\nND summary:")
print(df_new["ND"].describe())


##### Why is TSD usually higher than ND?
Because TSD includes ND plus additional loads/requirements, typically:
- Station demand / auxiliary load (power used by power stations themselves)
- Pumped storage pumping load (when pumped storage is consuming electricity to pump water uphill)
- Net exports through interconnectors (if GB is exporting, the system must generate extra to supply that export)

In [ ]:
# Consistency check: TSD vs ND where usually TSD >= ND
rate = (df_new["TSD"] < df_new["ND"]).mean() * 100
print(f"% of rows where TSD < ND: {rate:.2f}%")

print("Example rows where TSD < ND:")
print(df_new[df_new["TSD"] < df_new["ND"]][["SETTLEMENT_DATE","SETTLEMENT_PERIOD","ND","TSD"]].head(20))

In [ ]:
df_new.dtypes

In [ ]:
df_final = df_new[["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "ND", "TSD"]].copy()

In [ ]:
dups = df_final[df_final.duplicated(["SETTLEMENT_DATE","SETTLEMENT_PERIOD"], keep=False)] \
    .sort_values(["SETTLEMENT_DATE","SETTLEMENT_PERIOD"])

print("Number of duplicate (date, period) rows:", dups.shape[0])
print("Example duplicate rows:")
print(dups.head(20))

In [ ]:
print(df_final[["SETTLEMENT_DATE","SETTLEMENT_PERIOD","ND","TSD"]].dtypes)

 ### BUILD CANONICAL UTC TIMESTAMPS FROM SETTLEMENT STRUCTURE ###

In [ ]:
# Sort for stability
df_final = df_final.sort_values(["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]).reset_index(drop=True)


In [ ]:
# Quality check of rows per settlement date
rows_per_day = df_final.groupby("SETTLEMENT_DATE").size()
print("Rows per settlement date distribution:")
print(rows_per_day.value_counts().sort_index())

In [ ]:
# Checking each settlement date should contain 46, 48, or 50 half-hour rows
# (46/50 occur on DST transition days; 48 is a normal day)

unexpected_days = rows_per_day[~rows_per_day.isin([46, 48, 50])]
print("\nUnexpected settlement-date row counts:", len(unexpected_days))

# Flag any dates with unexpected row counts
if len(unexpected_days) > 0:
    print(unexpected_days.head(20))

In [ ]:
# Created a timezone-aware local-midnight timestamp for each settlement date
# This gives a safe start-of-day anchor in UK local time (Europe/London)
# from which settlement periods can be offset in 30-minute steps
local_midnight = pd.DatetimeIndex(df_final["SETTLEMENT_DATE"]).tz_localize("Europe/London")


### Why this is needed? ###
We need a reliable starting point for each settlement day. Instead of trying to guess each half-hour’s local clock time directly, we anchor the day at local midnight, then count forward by settlement periods.

Example: If SETTLEMENT_DATE = 2024-07-01, this line creates: 2024-07-01 00:00:00 Europe/London
Then:
- settlement period 1 -> midnight + 0 minutes
- settlement period 2 -> midnight + 30 minutes
- settlement period 3 -> midnight + 60 minutes
 and so on.

That is safer because midnight itself is not ambiguous or missing, even on DST transition days.

In [ ]:
# Convert the Europe/London settlement-day anchor to UTC.
midnight_utc = local_midnight.tz_convert("UTC")

Example:
- If local_midnight = 2024-07-01 00:00 Europe/London (summer, so GB is on BST = UTC+1),
 then: midnight_utc = 2024-06-30 23:00 UTC

- If local_midnight = 2024-12-01 00:00 Europe/London (winter, so GB is on GMT = UTC+0),
 then: midnight_utc = 2024-12-01 00:00 UTC

In [ ]:
# Construct timestamp_utc from the UTC settlement-day anchor and settlement-period index.
# Each settlement period represents one consecutive 30-minute interval within the settlement day,
# so the timestamp is derived as:
# local-settlement-midnight converted to UTC + (settlement_period - 1) × 30 minutes
df_final["timestamp_utc"] = midnight_utc + pd.to_timedelta(
    (df_final["SETTLEMENT_PERIOD"] - 1) * 30,
    unit="m"
)

In [ ]:
df_final.head()

In [ ]:
print("\nHalf-hourly UTC range:")
print(df_final["timestamp_utc"].min(), "to", df_final["timestamp_utc"].max())


In [ ]:
# Check duplicate timestamp_utc at half-hour level
dup_halfhour = df_final.duplicated("timestamp_utc").sum()
print("\nDuplicate half-hour timestamp_utc rows:", dup_halfhour)

In [ ]:
df_final.head()

In [ ]:
df_final.tail()

In [ ]:
# Assign each half-hourly UTC timestamp to its containing UTC hour.
# This creates the hourly grouping key used to aggregate two half-hour rows into one hourly value.
df_final["hour_utc"] = df_final["timestamp_utc"].dt.floor("H")

 Example:
- 2024-07-01 10:00 UTC -> 2024-07-01 10:00 UTC
- 2024-07-01 10:30 UTC -> 2024-07-01 10:00 UTC
- 2024-07-01 11:00 UTC -> 2024-07-01 11:00 UTC

In [ ]:
df_final.head(10)

In [ ]:
#  Aggregate half-hourly demand data to hourly resolution.
hourly = (
    df_final.groupby("hour_utc", as_index=False)
    .agg(
        nd_mw=("ND", "mean"),
        tsd_mw=("TSD", "mean"),
        n_half_hours=("ND", "size")
    )
    .rename(columns={"hour_utc": "timestamp_utc"})
)

Note: It groups all rows that belong to the same hour_utc, and then for each hour it calculates:

- nd_mw = average of the half-hourly ND values in that hour
- tsd_mw = average of the half-hourly TSD values in that hour
- n_half_hours = how many half-hour rows were included in that hour

Then it renames hour_utc to timestamp_utc so the output matches the deliverable schema.

In [ ]:
print("\nHalf-hours per hourly bucket:")
print(hourly["n_half_hours"].value_counts().sort_index())

Note: Counts how many rows have:

n_half_hours = 1, 
n_half_hours = 2, 
n_half_hours = 3
and so on, then sorts them in numerical order.

In [ ]:
# QA check before reindexing:
# compare the observed hourly UTC timestamps against the full expected hourly range
# to confirm whether the aggregated series is contiguous or contains gaps
observed_hour_index = pd.DatetimeIndex(hourly["timestamp_utc"])

expected_hour_index = pd.date_range(
    start=observed_hour_index.min(),
    end=observed_hour_index.max(),
    freq="H",
    tz="UTC"
)

missing_hours_pre_reindex = expected_hour_index.difference(observed_hour_index)

print("Observed hourly rows:", len(observed_hour_index))
print("Expected hourly rows:", len(expected_hour_index))
print("Missing hourly timestamps before reindex:", len(missing_hours_pre_reindex))

if len(missing_hours_pre_reindex) > 0:
    print("\nFirst 20 missing hourly timestamps:")
    print(missing_hours_pre_reindex[:20])

In [ ]:
# Reindex to the full hourly UTC timeline as a defensive consistency check
full_hour_index = pd.date_range(
    start=hourly["timestamp_utc"].min(),
    end=hourly["timestamp_utc"].max(),
    freq="H",
    tz="UTC"
)

Note: Reindexing is retained as a defensive consistency step, but the pre-check confirms the hourly series is already contiguous before reindexing.

In [ ]:
# Reindex the aggregated hourly series onto the full expected hourly UTC timeline.
hourly = (
    hourly.set_index("timestamp_utc")
    .reindex(full_hour_index)
    .rename_axis("timestamp_utc")
    .reset_index()
)

### DELIVERABLE 1 ###

In [ ]:
# Save deliverable 1
hourly_output = hourly[["timestamp_utc", "nd_mw", "tsd_mw"]].copy()
hourly_output.to_parquet("demand_hist_hourly.parquet", index=False)

print("\nSaved demand_hist_hourly.parquet")
hourly_output.head()

In [ ]:
hourly_output.to_csv("demand_hist_hourly.csv", index=False)

In [ ]:
# Quality check: DST / LOCAL VS UTC DATE CHECKS

# 1: number of half-hour settlement periods per settlement date
# This confirms that local settlement days can validly contain 46, 48, or 50 half-hour periods.
periods_per_day = df_final.groupby("SETTLEMENT_DATE").size()

print("Settlement periods per settlement date:")
print(periods_per_day.value_counts().sort_index())

# 2: number of hourly observations per local GB date
# Convert UTC timestamps back to Europe/London to show that local dates can contain
# 23, 24, or 25 hours across DST transitions.
hourly_check = hourly_output.copy()
hourly_check["timestamp_local"] = hourly_check["timestamp_utc"].dt.tz_convert("Europe/London")
hourly_check["date_local"] = hourly_check["timestamp_local"].dt.date

hours_per_local_date = hourly_check.groupby("date_local").size()

print("\nHours per local GB date:")
print(hours_per_local_date.value_counts().sort_index())

# 3: number of hourly observations per UTC date
# This shows that the final UTC series remains a regular hourly timeline,
# even though local-date hour counts vary at DST transitions.
hourly_check["date_utc"] = hourly_check["timestamp_utc"].dt.date
hours_per_utc_date = hourly_check.groupby("date_utc").size()

print("\nHours per UTC date:")
print(hours_per_utc_date.value_counts().sort_index())



In [ ]:
hourly_output.shape

In [ ]:
from pathlib import Path

hourly_readme = """# demand_hist_hourly.parquet — README

## Overview
This deliverable contains an **hourly historical demand time series for Great Britain (GB)** derived from NESO “Historic Demand” half-hourly settlement data (2010-2024).

## Data summary
- **Input data:** NESO Historic Demand Data (half-hourly annual files)
- **Geographic scope:** Great Britain (England + Wales + Scotland; excludes Northern Ireland)
- **Date range:** 2010–2024
- **Primary source fields used:** `SETTLEMENT_DATE`, `SETTLEMENT_PERIOD`, `ND`, `TSD`
- **Missing Values:** 0
- **Total rows in final output:** 131496

**File:** `demand_hist_hourly.parquet`

**Columns:**
- `timestamp_utc` — timezone-aware UTC timestamp at hourly resolution
- `nd_mw` — hourly mean National Demand, in MW
- `tsd_mw` — hourly mean Transmission System Demand, in MW

---

## Source data and granularity
The NESO historic demand source data is indexed by:
- `SETTLEMENT_DATE` — settlement day in GB local time (`Europe/London`)
- `SETTLEMENT_PERIOD` — half-hour period number within the settlement day

Each settlement period represents a **30-minute** interval.

Valid settlement-day lengths are expected to be:
- **48 periods** on standard days
- **46 periods** on spring DST transition days
- **50 periods** on autumn DST transition days

---

## Column definitions

### ND — National Demand (MW)
**National Demand (ND)** is the core GB demand series used in this project and is treated as the primary demand variable for modelling and FES anchoring.

### TSD — Transmission System Demand (MW)
**Transmission System Demand (TSD)** is retained as a secondary GB demand series for comparison and downstream use.

### England & Wales demand
Some NESO historic demand files may include `ENGLAND_WALES_DEMAND`, which is an England & Wales-specific measure rather than a GB-wide series.  
This deliverable focuses on the GB-wide outputs `ND` and `TSD`, so `ENGLAND_WALES_DEMAND` is not included in the final parquet output.

---

## Time handling and timestamp construction

### Why timestamp construction is needed
The NESO source is supplied as settlement date plus settlement period, rather than as an explicit UTC timestamp.

### Settlement-date anchor
To construct timestamps robustly across DST transitions:
1. each `SETTLEMENT_DATE` is anchored at **local midnight** in `Europe/London`
2. that local-midnight anchor is converted to **UTC**
3. each half-hourly timestamp is then created as:

`timestamp_utc = local settlement midnight converted to UTC + (SETTLEMENT_PERIOD - 1) × 30 minutes`

This approach preserves valid **46-, 48-, and 50-period settlement days** without dropping rows and avoids ambiguity from constructing local wall-clock timestamps directly.

### DST handling
DST affects the number of periods in a **local settlement day**, but the final output is stored in **UTC**.

As a result:
- local GB dates may correspond to **23, 24, or 25 hours**
- the final `timestamp_utc` series is expected to remain **continuous and gap-free in UTC**

---

## Half-hourly to hourly aggregation method

### Objective
Convert half-hourly MW demand values into an hourly MW demand time series.

### Method
1. Each half-hourly `timestamp_utc` is assigned to its containing UTC hour:
   - `hour_utc = floor(timestamp_utc to the hour)`
2. Half-hourly values are aggregated within each UTC hour using the **mean**:
   - `nd_mw = mean(ND over the hour)`
   - `tsd_mw = mean(TSD over the hour)`

### Why mean?
ND and TSD are power values measured in **MW**.  
Taking the mean of the two half-hourly MW observations gives the **average MW across the hour**, which is the appropriate hourly representation for this deliverable.


---

## Units
- `ND`, `TSD`, `nd_mw`, and `tsd_mw` are in **megawatts (MW)**

---
"""

Path("demand_hist_hourly_README.md").write_text(hourly_readme, encoding="utf-8")
print("Saved: demand_hist_hourly_README.md")

### DELIVERABLE 2 ###

In [ ]:
# Aggregate half-hourly demand data to settlement-date daily summaries.
# For each settlement date:
# - nd_sum_mw is the sum of half-hourly ND values
# - nd_mean_mw is the mean half-hourly ND across the day
# - tsd_sum_mw is the sum of half-hourly TSD values
# The summed MW fields are intermediate quantities used to calculate daily MWh in the next step.
daily = (
    df_final.groupby("SETTLEMENT_DATE", as_index=False)
    .agg(
        nd_sum_mw=("ND", "sum"),
        nd_mean_mw=("ND", "mean"),
        tsd_sum_mw=("TSD", "sum")
    )
)

Example: If one settlement date has the following half-hourly ND values: [100, 120, 110, 130]
and TSD values: [140, 150, 145, 155]

then the daily summary for that date will be:
-   nd_sum_mw  = 100 + 120 + 110 + 130 = 460
-   nd_mean_mw = (100 + 120 + 110 + 130) / 4 = 115
-   tsd_sum_mw = 140 + 150 + 145 + 155 = 590

In [ ]:
# Convert daily summed half-hourly demand from MW-period units into MWh.
# Because each settlement period is 30 minutes long, multiply the daily sum by 0.5.
daily["nd_daily_mwh"] = daily["nd_sum_mw"] * 0.5
daily["tsd_daily_mwh"] = daily["tsd_sum_mw"] * 0.5

# Format the final daily output to match the required schema.
# SETTLEMENT_DATE is renamed to date and only deliverable columns are retained.
daily_output = daily.rename(columns={"SETTLEMENT_DATE": "date"})[
    ["date", "nd_daily_mwh", "nd_mean_mw", "tsd_daily_mwh"]
].copy()


### Why 0.5 exactly? ###

Because: 30 minutes = 0.5 hours
And the energy formula is: Energy (MWh) = Power (MW) × Time (hours)

So for each settlement period: energy for one row = MW × 0.5

If you sum many half-hour rows first, then multiply the total by 0.5, that gives total daily MWh.

Notes: The hourly deliverable is indexed by canonical UTC timestamps to provide a clean, contiguous hourly modelling series without DST ambiguity. The daily deliverable is grouped by SETTLEMENT_DATE, which represents the GB local settlement day in the NESO source data. This is intentional: hourly outputs prioritise time-series consistency in UTC, while daily outputs preserve the natural source-day definition used in settlement data. As a result, date in demand_hist_daily.parquet should be interpreted as a local settlement-date field rather than a UTC calendar date, and daily values should not be assumed to equal a simple UTC-date rollup of the hourly file around DST or date-boundary shifts.

In [ ]:
print("Daily row count:", len(daily_output))
print("Daily date range:", daily_output["date"].min(), "to", daily_output["date"].max())

In [ ]:
daily_output.to_parquet("demand_hist_daily.parquet", index=False)
print("Saved demand_hist_daily.parquet")

In [ ]:
daily_output.head()

In [ ]:
daily_output.to_csv("demand_hist_daily.csv", index=False)

In [ ]:
daily_output.shape

In [ ]:
daily_readme = """# demand_hist_daily.parquet — README

## Overview
This deliverable contains **daily historical demand summaries for Great Britain (GB)** derived from NESO “Historic Demand” half-hourly settlement data.

## Data summary
- **Input data:** NESO Historic Demand Data (half-hourly annual files)
- **Geographic scope:** Great Britain (England + Wales + Scotland; excludes Northern Ireland)
- **Date range:** 2010–2024
- **Primary source fields used:** `SETTLEMENT_DATE`, `SETTLEMENT_PERIOD`, `ND`, `TSD`
- **Missing Values:** 0
- **Total rows in final output:** 5479

**File:** `demand_hist_daily.parquet`

**Columns:**
- `date` — settlement date (GB local settlement date)
- `nd_daily_mwh` — daily National Demand energy, in MWh
- `nd_mean_mw` — mean half-hourly National Demand over the settlement day, in MW
- `tsd_daily_mwh` — daily Transmission System Demand energy, in MWh

---

## Source data and granularity
The NESO historic demand source data is indexed by:
- `SETTLEMENT_DATE` — settlement day in GB local time (`Europe/London`)
- `SETTLEMENT_PERIOD` — half-hour period number within the settlement day

Each settlement period represents a **30-minute** interval.

Valid settlement-day lengths are expected to be:
- **48 periods** on standard days
- **46 periods** on spring DST transition days
- **50 periods** on autumn DST transition days

---

## Column definitions

### ND — National Demand (MW)
**National Demand (ND)** is the core GB demand series used in this project and is treated as the primary demand variable for modelling and FES anchoring.

### TSD — Transmission System Demand (MW)
**Transmission System Demand (TSD)** is retained as a secondary GB demand series for comparison and downstream use.

---

## Daily calculations

### Daily aggregation basis
The daily file is derived directly from the cleaned half-hourly demand dataset using:
- `SETTLEMENT_DATE`
- `SETTLEMENT_PERIOD`
- `ND`
- `TSD`

The daily output is grouped by **settlement date**, so `date` represents the GB local settlement day rather than a UTC date.
- `the hourly deliverable is indexed by timestamp_utc`
- `the daily deliverable is grouped by SETTLEMENT_DATE`. A daily aggregation should usually respect the natural day boundary of the source system. Here, the source system is settlement-based, not UTC-calendar-based.

### Daily mean demand (MW)
Daily mean demand is calculated from the half-hourly MW values as:

- `nd_mean_mw = mean(ND over all settlement periods in that date)`

This represents the average ND power level across that settlement day.

### Daily energy (MWh) from half-hourly MW
Daily energy is calculated from half-hourly power values using the 30-minute settlement-period duration:

- `nd_daily_mwh = sum(ND over all periods in that date) × 0.5`
- `tsd_daily_mwh = sum(TSD over all periods in that date) × 0.5`

### Why multiply by 0.5?
ND and TSD are measured in **MW**, which is power.  
To convert power to energy:

- energy (MWh) = power (MW) × time (hours)

Because each settlement period covers **0.5 hours**, the summed half-hourly MW values are multiplied by **0.5** to derive daily energy in MWh.

---

## DST considerations
DST affects the number of settlement periods in a **local GB settlement day**:
- standard days contain 48 half-hour periods
- spring transition days contain 46 periods
- autumn transition days contain 50 periods

Here, these valid settlement-day lengths are preserved rather than removed during timestamp construction.  
As a result, daily calculations are based on the full valid set of settlement periods available for each settlement date.

This means:
- daily mean demand is computed across the actual settlement periods present on that date
- daily energy is computed using the full valid half-hourly coverage for that settlement date


---

## Units
- `nd_mean_mw` is in **MW**
- `nd_daily_mwh` is in **MWh**
- `tsd_daily_mwh` is in **MWh**

---


"""

Path("demand_hist_daily_README.md").write_text(daily_readme, encoding="utf-8")
print("Saved: demand_hist_daily_README.md")

### DELIVERABLE 3 ###

Note: This shape library is based on hourly ND values and will be used to capture typical within-day demand patterns by month, day type, and hour.


In [ ]:
holiday_calendar = pd.read_csv("holiday_calendar_2010_2045.csv")
holiday_calendar["date"] = pd.to_datetime(holiday_calendar["date"])

holiday_calendar_hist = holiday_calendar[
    holiday_calendar["date"].between("2010-01-01", "2024-12-31")
].copy()

assert holiday_calendar_hist["date"].duplicated().sum() == 0
assert holiday_calendar_hist[
    ["is_holiday_eng_wales", "is_holiday_scotland", "is_holiday_gb_any"]
].isna().sum().sum() == 0

In [ ]:
# DELIVERABLE 3: HOLIDAY-AWARE HOURLY SHAPE LIBRARY

hourly2 = hourly_output.copy()

# Convert UTC timestamps to Europe/London local time so shape features
# reflect local GB demand patterns.
hourly2["timestamp_local"] = hourly2["timestamp_utc"].dt.tz_convert("Europe/London")
hourly2["date_local"] = pd.to_datetime(hourly2["timestamp_local"].dt.date)
hourly2["month"] = hourly2["timestamp_local"].dt.month.astype(int)
hourly2["hour"] = hourly2["timestamp_local"].dt.hour.astype(int)
hourly2["day_of_week"] = hourly2["timestamp_local"].dt.dayofweek.astype(int)

# Bring in GB holiday flag using local GB date.
hourly2 = hourly2.merge(
    holiday_calendar_hist[["date", "is_holiday_gb_any"]],
    left_on="date_local",
    right_on="date",
    how="left",
    validate="many_to_one",
)

missing_holiday_flags = hourly2["is_holiday_gb_any"].isna().sum()
assert missing_holiday_flags == 0, f"Missing holiday flags: {missing_holiday_flags}"

hourly2["is_holiday_gb_any"] = hourly2["is_holiday_gb_any"].astype(int)

# Holiday takes priority over weekday/weekend.
hourly2["day_type"] = np.select(
    [
        hourly2["is_holiday_gb_any"].eq(1),
        hourly2["day_of_week"].ge(5),
    ],
    [
        "holiday",
        "weekend",
    ],
    default="weekday",
)

# Compute local-day ND totals and convert each hour into a fraction of its local-day total.
hourly2["nd_daily_total_local"] = (
    hourly2.groupby("date_local")["nd_mw"].transform("sum")
)

hourly2["nd_hour_fraction_raw"] = (
    hourly2["nd_mw"] / hourly2["nd_daily_total_local"]
)

# Average hourly fractions by month, day type, and hour to form the shape library.
shape_library = (
    hourly2
    .groupby(["month", "day_type", "hour"], as_index=False)
    .agg(nd_hour_fraction=("nd_hour_fraction_raw", "mean"))
    .sort_values(["month", "day_type", "hour"])
    .reset_index(drop=True)
)

# Re-normalise so each month × day_type profile sums exactly to 1.
shape_library["nd_hour_fraction"] = (
    shape_library["nd_hour_fraction"]
    / shape_library.groupby(["month", "day_type"])["nd_hour_fraction"].transform("sum")
)

print("Shape library rows:", len(shape_library))
print("Day types:", sorted(shape_library["day_type"].unique()))

shape_library.head(24)

In [ ]:
# check the sum is equal to 1
shape_library["nd_hour_fraction"] = (
    shape_library["nd_hour_fraction"]
    / shape_library.groupby(["month", "day_type"])["nd_hour_fraction"].transform("sum")
)

In [ ]:
# QA checks for holiday-aware shape library

assert shape_library.duplicated(["month", "day_type", "hour"]).sum() == 0

shape_check = (
    shape_library
    .groupby(["month", "day_type"], as_index=False)
    ["nd_hour_fraction"]
    .sum()
)

print(shape_check)

max_fraction_error = (shape_check["nd_hour_fraction"] - 1.0).abs().max()
print("Max fraction-sum error:", max_fraction_error)

assert max_fraction_error < 1e-10

print("Rows by day_type:")
print(shape_library["day_type"].value_counts())

print("Hours per month/day_type:")
print(
    shape_library
    .groupby(["month", "day_type"])["hour"]
    .nunique()
    .unstack(fill_value=0)
)

In [ ]:
shape_library = shape_library[
    ["month", "day_type", "hour", "nd_hour_fraction"]
].copy()

shape_library.to_csv("demand_hourly_shape_library.csv", index=False)
shape_library.to_parquet("demand_hourly_shape_library.parquet", index=False)

print("Saved holiday-aware demand_hourly_shape_library.csv")
print("Saved holiday-aware demand_hourly_shape_library.parquet")

In [ ]:
from pathlib import Path

shape_readme = """# demand_hourly_shape_library.parquet — README

## Overview
This deliverable is an **hourly demand shape library** used to redistribute **daily demand values into hourly profiles** for Great Britain (GB).

## Data summary
- **Input basis:** cleaned historical hourly demand output derived from NESO Historic Demand data
- **Geographic scope:** Great Britain (England + Wales + Scotland; excludes Northern Ireland)
- **Date range:** 2010–2024
- **Primary shaping variable:** `nd_mw` (National Demand)

**File:** `demand_hourly_shape_library.parquet`

**Columns:**
- `month` — month number (1–12) based on GB local time
- `day_type` — `weekday`, `weekend`, or `holiday`
- `hour` — hour of day (0–23) in GB local time
- `nd_hour_fraction` — average fraction of a local day’s demand occurring in that hour (unitless)

---

## Purpose
Forecasting or scenario modelling is often easier at **daily** resolution.  
This library supports conversion from **daily demand → hourly demand** by providing a typical within-day ND profile conditioned on:
- **month** (captures seasonal variation)
- **day type** (captures weekday, weekend, and GB bank-holiday behaviour)
- **hour** (captures intraday timing patterns)

The output is intended for downstream future hourly demand reconstruction rather than direct reproduction of historical absolute demand levels.

---

## Input basis
The library is built from the final cleaned hourly historical demand output, which contains:
- `timestamp_utc`
- `nd_mw`
- `tsd_mw`
- The shape library also uses holiday_calendar_2010_2045.csv to identify GB bank holidays during the historical calibration period.

Only **ND** is used to construct the shape library, because ND is the primary demand definition used in this project.

---

## How the library is constructed

### Step 1: Start from hourly ND series
The shape library starts from the cleaned hourly ND series (`nd_mw`) indexed by `timestamp_utc`.

### Step 2: Convert UTC timestamps to GB local time
Although the hourly demand file is stored in **UTC**, demand shape behaviour is driven by **local GB time**.  
Therefore, `timestamp_utc` is converted back to `Europe/London` to derive:
- `date_local`
- `month`
- `hour`
- `day_of_week`
- `day_type`

This ensures that hourly shape features align with local calendar behaviour rather than UTC clock time.

### Step 3: Compute local-day ND totals
For each **local GB date**, total daily ND is calculated as:

- `nd_daily_total_local = sum(nd_mw across all hours in that local day)`

Because `nd_mw` is an hourly MW series, summing across the hours of a local day gives the daily total in MWh-equivalent terms for shaping purposes.

### Step 4: Compute hourly fraction within each local day
For each hour within a local day:

- `nd_hour_fraction = nd_mw / nd_daily_total_local`

This produces a **unitless within-day fraction**.  
For example, a value of `0.035` means that approximately **3.5%** of that local day’s total demand occurs in that hour.

### Step 5: Average fractions into the reusable library
Hourly fractions are averaged across the historical period by:
- `month`
- `day_type`
- `hour`

This produces a typical hourly demand shape for each month × day-type × hour combination.

---

## Day type definition

- `holiday` = any date where `is_holiday_gb_any == 1`
- `weekend` = Saturday or Sunday where `is_holiday_gb_any == 0`
- `weekday` = Monday to Friday where `is_holiday_gb_any == 0`

Holiday classification takes priority over weekday/weekend classification. For example, a bank holiday that falls on a Monday is labelled `holiday`, not `weekday`.

Bank holidays are **not** included as a separate category in this version of the library.  

---

## DST and local-time considerations
The hourly source file is stored in **UTC**, but the shape library is intentionally built using **local GB time** because demand patterns follow local clock time.

As a result:
- local-date totals are based on `Europe/London`
- DST effects are handled through the UTC → local conversion step
- the shape library reflects observed local intraday behaviour rather than UTC-hour behaviour


---

## Units
- `nd_hour_fraction` is **unitless**

---

## QA
The library is grouped by `month`, `day_type`, and `hour`. Each `month × day_type` hourly profile is normalised so that `nd_hour_fraction` sums to 1. Some months may not contain a `holiday` profile if there were no historical GB holiday dates in that month.

"""

Path("demand_hourly_shape_library_README.md").write_text(shape_readme, encoding="utf-8")
print("Saved: demand_hourly_shape_library_README.md")